# 01 — Data Acquisition

**Week 1 of the [UAE Real Estate Market Intelligence](../README.md) build.**

This notebook is a **runbook**, not an exploration. Each cell does one verifiable thing, prints a row count or check, and moves on. By the end of this notebook:

- `real_estate.db` exists with schema applied
- `transactions` table has the DLD rows you downloaded
- `macro_indicators` has UAE rates + Brent + CPI
- All Week 1 exit criteria pass

If you haven't yet, follow the **Setup** section of the README first — `pip install -r requirements.txt` and choose one of the three data paths (A: manual download, B: Playwright, C: Kaggle mirror).

## 0. Setup — make `src` importable from a notebook subfolder

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

from src import config, db, load_dld, macro
print('project root:', ROOT)
print('db path     :', config.DB_PATH.name)

## 1. Initialize the SQLite schema
Idempotent — safe to re-run.

In [ ]:
db.init_schema()
with db.connect() as conn:
    tables = [r['name'] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
tables

## 2. Confirm raw CSVs are in place

The loader reads from `data/raw/`. Drop your DLD CSVs there before running the next cell. If `data/raw/` is empty:

- **Path A**: download from <https://dubailand.gov.ae/en/open-data/real-estate-data/> (set From → To, click Download as CSV), drop here.
- **Path B**: `python -m src.scrape_dld --years 2024 --dataset transactions`
- **Path C**: any Kaggle DLD mirror unzipped into `data/raw/`.

In [ ]:
raw_files = sorted(config.RAW_DIR.glob('*.csv'))
for p in raw_files:
    size_mb = p.stat().st_size / 1024 / 1024
    print(f'  {p.name:<40} {size_mb:>8.1f} MB')
print(f'\n{len(raw_files)} CSV(s) found in {config.RAW_DIR}')
assert raw_files, f'No CSVs in {config.RAW_DIR}. See section 2 above.'

## 3. Peek at the first CSV — confirm column auto-detection works

If the loader can't resolve required columns, this is where you'll see it. The column synonym registry lives in `src/load_dld.py` — add new synonyms there if your source has unusual names.

In [ ]:
sample_path = raw_files[0]
head = pd.read_csv(sample_path, nrows=0, encoding_errors='ignore')
print('source columns:', list(head.columns))
detected = load_dld._detect_table(list(head.columns))
mapping  = load_dld._resolve_columns(
    list(head.columns),
    load_dld.TXN_SYNONYMS if detected == 'transactions' else load_dld.RENT_SYNONYMS,
)
print(f'\ndetected table: {detected}')
print(f'resolved {len(mapping)} columns:')
for canonical, src in mapping.items():
    print(f'  {canonical:<22} <- {src}')

## 4. Run the loader on everything in `data/raw/`

In [ ]:
new_rows = load_dld.load_path(config.RAW_DIR)
print(f'inserted {new_rows:,} new rows total')

## 5. Sanity-check row counts and date coverage

In [ ]:
with db.connect() as conn:
    txn_count = conn.execute('SELECT COUNT(*) AS n FROM transactions').fetchone()['n']
    date_span = conn.execute(
        'SELECT MIN(txn_date) AS d_min, MAX(txn_date) AS d_max FROM transactions'
    ).fetchone()
    by_year = pd.read_sql(
        """
        SELECT substr(txn_date,1,4) AS year, COUNT(*) AS rows,
               printf('%.0f', AVG(price_aed)) AS avg_price_aed
        FROM transactions GROUP BY 1 ORDER BY 1
        """,
        conn,
    )
    areas_count = conn.execute('SELECT COUNT(*) AS n FROM areas').fetchone()['n']

print(f'transactions : {txn_count:>10,}')
print(f'date span    : {date_span["d_min"]} → {date_span["d_max"]}')
print(f'areas (dim)  : {areas_count:>10,}')
by_year

## 6. Pull macro indicators (rates, Brent, CPI)

In [ ]:
macro.refresh_all()
with db.connect() as conn:
    summary = pd.read_sql(
        """
        SELECT indicator, COUNT(*) AS rows,
               MIN(obs_date) AS d_min, MAX(obs_date) AS d_max,
               printf('%.2f', AVG(value)) AS mean_value
        FROM macro_indicators GROUP BY indicator
        """,
        conn,
    )
summary

## 7. Week 1 exit-criteria gate

These assertions match the [build plan](../../../.claude/plans/you-are-an-expert-linked-wombat.md). If anything fails, fix before moving to Week 2.

In [ ]:
with db.connect() as conn:
    txn_count   = conn.execute('SELECT COUNT(*) FROM transactions').fetchone()[0]
    macro_count = conn.execute('SELECT COUNT(*) FROM macro_indicators').fetchone()[0]

checks = [
    ('transactions row count >= 500,000', txn_count >= 500_000, f'{txn_count:,}'),
    ('macro_indicators populated',        macro_count > 0,       f'{macro_count:,}'),
    ('areas dimension populated',         areas_count > 0,       f'{areas_count:,}'),
]
width = max(len(c[0]) for c in checks)
for desc, ok, actual in checks:
    mark = 'PASS' if ok else 'FAIL'
    print(f'  [{mark}] {desc:<{width}}  actual: {actual}')

if not all(c[1] for c in checks):
    print('\nSome checks failed. Common fixes:')
    print('  - Need more years of CSV in data/raw/ for the 500k threshold')
    print('  - If macro_indicators is empty, check internet access + run macro.refresh_all() again')

---
**Next:** open [`02_cleaning.ipynb`](02_cleaning.ipynb) for Week 2 — area normalization, geocoding, outlier flagging, and derived metrics (price/sqft, rolling medians, YoY, rental yield).